These are my notes post the let's build gpt video.
## Self-Attention block:
We basically have in an attention block: 
- an attention sub-block: x1 = softmax(Q K^T / sqrt (d_k) + M ) V 
- adding the residual (x = x + x1) 
- an MLP feedforward sub-block : x2 
- adding the residual x = x + x2. 

in karpathy's vid, he did layer norm before the attention sub-block and added a dropout at after the second risidual.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F



In the following, we suppose we have 4 heads in our attention block, each token is represented by a 32-dimensional embeding vector, our max token window is 12 and we have 8 batches.

In [2]:
# setting:
B,H,T,n_embed = 8,4,12,32
d_k = n_embed // H
#x here is the input to our self-attention subblock.
x = torch.randn([B,T,n_embed]) #B,H,T,n_embed

print(x.shape)

Q = nn.Linear(n_embed, H*d_k, bias=False)
K = nn.Linear(n_embed, H*d_k, bias=False)
V = nn.Linear(n_embed, H*d_k, bias=False)

query = Q(x)
query = query.view(B,T,H,d_k).transpose(1,2)
key = K(x)
key = key.view(B,T,H,d_k).transpose(1,2)
value = V(x)
value = value.view(B,T,H,d_k).transpose(1,2)
#it's okay to alter the views here because the exact position of the weights isn't important, they'll adjust.


print (query.shape, key.shape, value.shape)

# we now calculate the product:
prod1 = query @ key.transpose(-2,-1) / (d_k)**(1/2)
# we create the masc:
tril = torch.tril(torch.ones(T,T))
masc = tril.masked_fill(tril == 0 , -float("inf"))
masc = masc.masked_fill(masc == 1 , 0)
# we sum and apply the soft max:
print(prod1.shape, masc.shape, value.shape)
x1 = F.softmax(prod1 + masc, dim = -1) @ value
print(x1.shape)




#Now we lay the W_O subblock and the risidual: 
W_O = nn.Linear(n_embed, n_embed )
x = x + W_O(x1.transpose(1,2).reshape(B,T,n_embed))

# now for the feed forward:
FF = nn.Sequential(nn.Linear(n_embed, 4*n_embed), nn.GELU(), nn.Linear(4 * n_embed, n_embed))
x = FF(x) + x

#(intentionally skiped the layernorm for now)


torch.Size([8, 12, 32])
torch.Size([8, 4, 12, 8]) torch.Size([8, 4, 12, 8]) torch.Size([8, 4, 12, 8])
torch.Size([8, 4, 12, 12]) torch.Size([12, 12]) torch.Size([8, 4, 12, 8])
torch.Size([8, 4, 12, 8])


when doing Q = nn.Linear(n_embed, d_k, bias=False) we're not really dividing the vector of imbeding into 4 parts exactly but more like, creating linear transformations to four 